## Clean & Merge — World Cup 2026 Dataset

This notebook builds one merged, model-ready dataset from three raw sources:

1. **results.csv** — historical match results (filtered to last 25 years)
2. **fifa_ranking-*.csv** — FIFA ranking history 
3. **wc_team_features_2026** — bonus team-level features for 2026 (squad value, history, etc.)

In [28]:
from pathlib import Path
import pandas as pd

RAW_DATA = Path("../data/raw")
PROCESSED_DATA = Path("../data/processed")

In [29]:
RESULTS_DATE_COL = "date"
RESULTS_HOME_COL = "home_team"
RESULTS_AWAY_COL = "away_team"
RESULTS_HOME_SCORE_COL = "home_score"
RESULTS_AWAY_SCORE_COL = "away_score"

RANKING_TEAM_COL = "country_full"
RANKING_DATE_COL = "rank_date"
RANKING_RANK_COL = "rank"
RANKING_POINTS_COL = "total_points"

# Only keeping matches from the last N years — old matches add noise more
# than signal for predicting 2026.
LOOKBACK_YEARS = 25

In [30]:
# As different dataset have same country name in different forms.
def standardize_team_name(name: str) -> str:
    if not isinstance(name, str):
        return name
    name = name.strip()
    overrides = {
        "United States": "USA",
        "United States of America": "USA",
        "IR Iran": "Iran",
        "Korea Republic": "South Korea",
        "Korea DPR": "North Korea",
        "China PR": "China",
        "Côte d'Ivoire": "Ivory Coast",
        "Czechia": "Czech Republic",
    }
    return overrides.get(name, name)


### Load Results dataset

In [31]:
df = pd.read_csv(RAW_DATA/'historical_matches'/'results.csv')
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [32]:
df[RESULTS_DATE_COL] = pd.to_datetime(df[RESULTS_DATE_COL], errors='coerce')
df[RESULTS_HOME_COL] = df[RESULTS_HOME_COL].apply(standardize_team_name)
df[RESULTS_AWAY_COL] = df[RESULTS_AWAY_COL].apply(standardize_team_name)
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [33]:
df['result'] = 'D'
df.loc[df[RESULTS_HOME_SCORE_COL] > df[RESULTS_AWAY_SCORE_COL], 'result'] = 'H'
df.loc[df[RESULTS_HOME_SCORE_COL] < df[RESULTS_AWAY_SCORE_COL], 'result'] = 'A'
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,D
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,H
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,H
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,D
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,H


In [34]:
# filtering matches, to only keep those from the last N years
cutoff_date = pd.Timestamp.now() - pd.DateOffset(years=LOOKBACK_YEARS)
results = df[df[RESULTS_DATE_COL] >= cutoff_date].reset_index(drop=True)
print(f"old results shape: {df.shape}")
print(f"new results shape: {results.shape}")
results.head()

old results shape: (49505, 10)
new results shape: (23710, 10)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result
0,2001-07-15,Brazil,Peru,2.0,0.0,Copa América,Cali,Colombia,True,H
1,2001-07-15,Congo,Ivory Coast,1.0,1.0,FIFA World Cup qualification,Pointe-Noire,Congo,False,D
2,2001-07-15,DR Congo,Tunisia,0.0,3.0,FIFA World Cup qualification,Kinshasa,DR Congo,False,A
3,2001-07-15,Ghana,Sudan,1.0,0.0,FIFA World Cup qualification,Accra,Ghana,False,H
4,2001-07-15,Paraguay,Mexico,0.0,0.0,Copa América,Cali,Colombia,True,D


In [35]:
results['result'].value_counts()

result
H    11372
A     6801
D     5537
Name: count, dtype: int64

### Load FIFA Ranking Dataset

In [36]:
ranking_dir = RAW_DATA/'fifa_rankings'
ranking_files = sorted(ranking_dir.glob("fifa_ranking-*.csv"))
ranking_files


[WindowsPath('../data/raw/fifa_rankings/fifa_ranking-2023-07-20.csv'),
 WindowsPath('../data/raw/fifa_rankings/fifa_ranking-2024-04-04.csv'),
 WindowsPath('../data/raw/fifa_rankings/fifa_ranking-2024-06-20.csv')]

In [37]:
print(pd.read_csv(ranking_files[0]).shape)
print(pd.read_csv(ranking_files[1]).shape)
print(pd.read_csv(ranking_files[2]).shape)

(64757, 8)
(67261, 8)
(67472, 8)


In [38]:
frames = []
for f in ranking_files:
    r = pd.read_csv(f)
    r[RANKING_DATE_COL] = pd.to_datetime(r[RANKING_DATE_COL], errors='coerce')
    frames.append(r)

rankings = pd.concat(frames, ignore_index=True)
rankings[RANKING_TEAM_COL] = rankings[RANKING_TEAM_COL].apply(standardize_team_name)

rankings = rankings.sort_values(RANKING_DATE_COL).drop_duplicates(subset=[RANKING_TEAM_COL, RANKING_DATE_COL], keep='last')
rankings.info()

<class 'pandas.DataFrame'>
Index: 69709 entries, 132076 to 199489
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   rank             69700 non-null  float64       
 1   country_full     69709 non-null  str           
 2   country_abrv     69709 non-null  str           
 3   total_points     69709 non-null  float64       
 4   previous_points  69709 non-null  float64       
 5   rank_change      69709 non-null  int64         
 6   confederation    69709 non-null  str           
 7   rank_date        69709 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(3), int64(1), str(3)
memory usage: 4.8 MB


In [39]:
# Taking only last 25 years data
ranking_cutoff = pd.Timestamp.now() - pd.DateOffset(years=LOOKBACK_YEARS)
rankings = rankings[rankings[RANKING_DATE_COL] >= ranking_cutoff]
rankings.shape

(52987, 8)

### Attach Rankings to each Match

In [40]:
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result
0,2001-07-15,Brazil,Peru,2.0,0.0,Copa América,Cali,Colombia,True,H
1,2001-07-15,Congo,Ivory Coast,1.0,1.0,FIFA World Cup qualification,Pointe-Noire,Congo,False,D
2,2001-07-15,DR Congo,Tunisia,0.0,3.0,FIFA World Cup qualification,Kinshasa,DR Congo,False,A
3,2001-07-15,Ghana,Sudan,1.0,0.0,FIFA World Cup qualification,Accra,Ghana,False,H
4,2001-07-15,Paraguay,Mexico,0.0,0.0,Copa América,Cali,Colombia,True,D


In [41]:
rankings.head()

,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
15888,184.0,Sao Tome e Principe,STP,110.0,111.0,0,CAF,2001-07-18
15934,158.0,Gambia,GAM,215.0,209.0,-2,CAF,2001-07-18
15920,128.0,St. Lucia,LCA,327.0,327.0,0,CONCACAF,2001-07-18
15908,162.0,Cape Verde Islands,CPV,206.0,208.0,1,CAF,2001-07-18
15918,130.0,St. Vincent and the Grenadines,VIN,323.0,324.0,0,CONCACAF,2001-07-18


In [42]:
def merge_side(results, team_col, prefix):
    side  = results[[RESULTS_DATE_COL, team_col]].rename(columns={team_col: RANKING_TEAM_COL})
    side_sorted = side.sort_values(RESULTS_DATE_COL)
    merged = pd.merge_asof(
        side_sorted,
        rankings.sort_values(RANKING_DATE_COL)[[RANKING_DATE_COL, RANKING_TEAM_COL, RANKING_RANK_COL, RANKING_POINTS_COL]],
        left_on = RESULTS_DATE_COL,
        right_on = RANKING_DATE_COL,
        by = RANKING_TEAM_COL,
        direction="backward"
    )
    merged = merged.rename(columns={
        RANKING_RANK_COL: f"{prefix}_rank",
        RANKING_POINTS_COL: f"{prefix}_points"
    })
    return merged[[f"{prefix}_rank", f"{prefix}_points"]]

In [43]:
home_ranks = merge_side(results, RESULTS_HOME_COL, "home")
away_ranks = merge_side(results, RESULTS_AWAY_COL, "away")

matches = results.reset_index(drop=True)
matches = pd.concat([matches, home_ranks.reset_index(drop=True), away_ranks.reset_index(drop=True)], axis=1)
matches["rank_diff"] = matches["away_rank"] - matches["home_rank"]
matches.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,home_rank,home_points,away_rank,away_points,rank_diff
19415,2022-06-04,Gambia,South Sudan,1.0,0.0,African Cup of Nations qualification,Thiès,Senegal,True,H,57.0,1406.87,59.0,1388.63,2.0
9845,2012-02-24,Singapore,Azerbaijan,2.0,2.0,Friendly,Dubai,United Arab Emirates,True,D,50.0,589.00,116.0,290.00,66.0
20039,2023-01-16,Thailand,Vietnam,1.0,0.0,AFF Championship,Pathum Thani,Thailand,False,H,111.0,1173.40,96.0,1228.63,-15.0
17510,2019-11-19,Burundi,Morocco,0.0,3.0,African Cup of Nations qualification,Bujumbura,Burundi,False,A,98.0,1236.00,NaN,NaN,NaN
21674,2024-07-13,Canada,Uruguay,2.0,2.0,Copa América,Charlotte,United States,True,D,48.0,1461.74,14.0,1663.44,-34.0


In [44]:
print(f"Missing values in home_rank: \n{home_ranks.isnull().sum()}")
print(f"Missing values in away_rank: \n{away_ranks.isnull().sum()}")

Missing values in home_rank: 
home_rank      1778
home_points    1775
dtype: int64
Missing values in away_rank: 
away_rank      1920
away_points    1914
dtype: int64


### Load 2026 Team Features Dataset

In [45]:
feat_dir = RAW_DATA/'wc_team_features_2026'
frames = []
for f in feat_dir.glob("*csv"):
    df = pd.read_csv(f)
    df['team'] = df['team'].apply(standardize_team_name)
    frames.append(df)

team_features = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"Team features shape: {team_features.shape}")
team_features.head()

Team features shape: (240, 24)


,version,team,continent,is_host,goals_scored_last_4y,goals_received_last_4y,wins_last_4y,losses_last_4y,draws_last_4y,world_cup_titles_before,...,world_cup_participations_before,groups_passed_before,round16_before,quarterfinals_before,semifinals_before,finals_before,winner,finalist,semi_finalist,quarter_finalist
0,2026,France,Europe,0,85,32,25,6,7,2,...,16,10,8,9,7,4,NaN,NaN,NaN,NaN
1,2026,Spain,Europe,0,104,32,29,2,8,1,...,16,11,9,6,2,1,NaN,NaN,NaN,NaN
2,2026,Argentina,South America,0,80,14,30,4,3,3,...,18,15,10,10,6,6,NaN,NaN,NaN,NaN
3,2026,England,Europe,0,82,23,26,6,7,1,...,16,13,8,10,3,1,NaN,NaN,NaN,NaN
4,2026,Portugal,Europe,0,98,31,26,5,7,0,...,8,5,4,3,2,0,NaN,NaN,NaN,NaN


### Saving Datasets

In [46]:
out_path = PROCESSED_DATA/'matches_merged.csv'
matches.to_csv(out_path, index=False)
print(f"Saved {out_path} — shape {matches.shape}")

feat_out_path = PROCESSED_DATA/'teams_features_2026.csv'
team_features.to_csv(feat_out_path, index=False)
print(f"Saved {feat_out_path} — shape {team_features.shape}")

Saved ..\data\processed\matches_merged.csv — shape (23710, 15)
Saved ..\data\processed\teams_features_2026.csv — shape (240, 24)
